# 02. Направление отказа

Этап 1. На `fp16` находим направление отказа и слой, на котором его абляция лучше всего снимает отказы. Для остальных конфигураций считаем косинус их собственного направления с `fp16`-направлением и AUROC.

`fp16` обязательно идёт первой. Используйте те же `LIMIT` и `RESULTS`, что и в ноутбуке 01.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO = "https://github.com/Kaban17/qsi.git"
if not os.path.exists("/content/qsi"):
    !git clone -q {REPO} /content/qsi
else:
    !git -C /content/qsi pull -q
%cd /content/qsi
!pip install -q -e . bitsandbytes accelerate
sys.path.insert(0, "/content/qsi")

In [ ]:
# Нужен секрет Colab с именем HF_TOKEN и принятая лицензия Gemma на Hugging Face.
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
from pathlib import Path
from transformers import AutoTokenizer
from qsi.data import load_splits, load_wikitext_chunks
from qsi.models import CONFIG_NAMES, MODEL_ID
from qsi.pipeline import truncate_splits
from qsi.results import read_metrics

# FULL = False: дымовой прогон на 5 промптах и двух конфигурациях (несколько минут).
# FULL = True: полный прогон всех 8 конфигураций. Значение должно совпадать во всех трёх ноутбуках.
FULL = False

LIMIT = None if FULL else 5
CONFIGS = CONFIG_NAMES if FULL else ["fp16", "rtn-4"]
RESULTS = Path("/content/drive/MyDrive/qsi") / ("results" if FULL else "results_smoke")

splits = truncate_splits(load_splits(), LIMIT)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
wikitext = load_wikitext_chunks(tokenizer, n_chunks=4 if LIMIT else 40)
{k: len(v) for k, v in splits.items()}, wikitext.shape

In [ ]:
import time
from qsi.pipeline import run_stage1

assert CONFIGS[0] == "fp16", "fp16 должна идти первой: с ней сравниваются остальные"
for name in CONFIGS:
    start = time.time()
    run_stage1(name, RESULTS, splits)
    print(f"{name}: {time.time() - start:.0f} s")

In [ ]:
import pandas as pd
pd.DataFrame({n: {k: read_metrics(RESULTS / n)["refusal"].get(k) for k in ("layer", "cos_selected", "auroc")} for n in CONFIGS}).T

In [ ]:
# Причинность направления: доля отказов на валидационных вредных промптах при абляции направления каждого слоя.
import matplotlib.pyplot as plt
r = read_metrics(RESULTS / "fp16")["refusal"]
plt.figure(figsize=(8, 3))
plt.plot(r["ablated_refusal_rate_per_layer"], marker="o", label="с абляцией")
plt.axhline(r["baseline_refusal_rate"], color="gray", linestyle="--", label="без абляции")
plt.xlabel("слой"); plt.ylabel("доля отказов"); plt.legend(); plt.title(f"Выбран слой {r['layer']}")
plt.show()